# 20 — Joint Forward-Cake Engine

The alternating engine (NB 01, NB 00's underlying `autocalibrate`) extracts a centroid per (ring, η) bin via the E-step, then minimises a residual on those centroids. That **discards the peak shape** — width, asymmetry, the η-dependent η-mixing — and depends on the centroid extractor being unbiased.

The **joint forward-cake** engine (`midas_calibrate_v2.pipelines.joint_cake`) refines geometry and a full pseudo-Voigt shape model **jointly against the raw 1-D radial intensity in each radial window**. There's no centroid step. The peak shape model is the original Wertheim form — independent σ_G, γ_L, η_v per (ring, η) bin — not the TCH polynomial.

Why this matters:

* **No centroid bias** — asymmetric peaks (cooler edges, doublet wings) bias centroids by hundreds of µε. The joint engine sees them as shape parameters, not strain.
* **Full Poisson weighting** — variance scales with intensity per the actual photon statistics, not a uniform per-fit weight.
* **Vectorised + windowed** — only the radial window around each ring is touched. On a 2880² image with 8 rings × 360 η bins × 16-pixel window, that's 46k forward evaluations per LBFGS step instead of 8M for the full cake.
* **All-differentiable** — geometry + 6 shape DOFs per (ring, η) refined together via L-BFGS in one pass.

Use it when:

* You're getting suspiciously consistent µε offsets you can't trace.
* You have doublet calibrants (Cu Kα₁/Kα₂) and the within-ring doublet bias is visible (see NB 08).
* You want a defensible per-(ring, η) report of peak width / mixing — useful for instrument-function PDF work.

**Don't** use it as your first pass — seed with the alternating engine (`calibrate()`), then escalate to joint-cake if the residuals warrant.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, h5py, time, numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

## 1. Load image + build a v1 params seed

The joint engine takes a v1 `CalibrationParams` plus the raw image, runs a couple of alternating iterations to seed geometry, then switches to L-BFGS over the joint (geometry, shape) problem.

In [ ]:
DATA = Path(os.environ.get('V2_ONESHOT_FILE',
    '/Users/hsharma/Desktop/analysis/leighanne/data/CeO2_XRD_echem_JLi_002587.vrx.h5'))
with h5py.File(DATA, 'r') as f:
    img  = f['exchange/data'][0].astype(np.float32)
    dark = f['exchange/data_dark'][0].astype(np.float32)

from midas_calibrate.params import CalibrationParams
v1 = CalibrationParams(
    NrPixelsY=2880, NrPixelsZ=2880, pxY=150.0, pxZ=150.0,
    # Seed from NB 00's known-good auto-seed result; the joint engine refines.
    Lsd=840_000.0, BC_y=1430.0, BC_z=1472.0,
    tx=0.0, ty=0.0, tz=0.0,
    Wavelength=0.184139, SpaceGroup=225,
    LatticeConstant=(5.4116, 5.4116, 5.4116, 90.0, 90.0, 90.0),
    MaxRingRad=2000.0, MinRingRad=120.0,
    Width=900.0,                   # ring half-width (µm) — sets window size
    EtaBinSize=2.0,                # finer eta bins help the joint shape model
    Refine={'Lsd': True, 'BC': True, 'ty': True, 'tz': True,
             'Wavelength': False, 'Parallax': False,
             **{f'p{i}': True for i in range(15)}},
    Device='cpu', Dtype='fp64',
)

## 2. Run the joint engine

Two stages internally:

1. **Seed** — `n_iter_seed=2` alternating iterations to get geometry close.
2. **Joint refinement** — build the cake at the seed, slice radial windows of `half_window_px` around each ring, add a per-(ring, η) `joint_shapes` Parameter of shape `[n_rings, n_eta, 6]` (σ, γ, η_v, area, bg0, bg1), L-BFGS jointly over everything for `lbfgs_max_iter` steps.

In [ ]:
from midas_calibrate_v2.pipelines.joint_cake import autocalibrate_joint

t0 = time.time()
result = autocalibrate_joint(
    v1, img, dark=dark,
    n_iter_seed=2,
    half_window_px=6.0,           # ±6 px window around each ring
    eta_bin_size_override=2.0,    # finer eta bins for shape DOFs
    lbfgs_max_iter=80,
    snr_min=3.0,                  # skip (ring, eta) bins with weak signal
    device='cpu', verbose=True,
)
print(f'\nwall time: {time.time()-t0:.1f} s')
print(f'final cost (sum of weighted squared residuals): {result.cost:.4e}')

## 3. The refined geometry

Same fields as the alternating engine's output — but they came from a *jointly* refined shape + geometry fit, so they're robust to centroid bias from asymmetric peaks.

In [ ]:
u = result.map_unpacked
print(f'GEOMETRY (joint cake MAP)')
print(f'  Lsd  = {float(u["Lsd"]):.3f} µm  ({float(u["Lsd"])/1000:.3f} mm)')
print(f'  BC   = ({float(u["BC_y"]):.4f}, {float(u["BC_z"]):.4f})')
print(f'  ty = {float(u["ty"]):+.5f}°    tz = {float(u["tz"]):+.5f}°')
print(f'\nDISTORTION (first 6)')
for n in ('iso_R2','iso_R4','iso_R6','a1','a2','a3'):
    if n in u:
        print(f'  {n:8s} = {float(u[n]):+.4e}')

## 4. The shape parameters

`joint_shapes` is a `[n_rings, n_eta, 6]` tensor: per-(ring, η) pseudo-Voigt with `(σ_G, γ_L, η_v, area, bg0, bg1)`. This is the **shape report** — directly inspectable, plottable, and exportable for downstream PDF / Rietveld instrument-function modelling.

In [ ]:
shapes = u['joint_shapes'].detach().cpu().numpy()    # [n_rings, n_eta, 6]
names = ['sigma_G', 'gamma_L', 'eta_v', 'area', 'bg0', 'bg1']
n_rings, n_eta, _ = shapes.shape
print(f'joint_shapes: {n_rings} rings × {n_eta} η bins × 6 params')
for i, name in enumerate(names[:3]):
    median_per_ring = np.median(shapes[..., i], axis=1)
    print(f'  {name:8s} median per-ring: {median_per_ring}')

In [ ]:
# Visualise σ_G vs eta for each ring — should be slowly-varying;
# spikes indicate problem regions (panel edges, doublet contamination).
fig, ax = plt.subplots(figsize=(11, 4.5))
eta = np.linspace(-180, 180, n_eta, endpoint=False)
for k in range(min(n_rings, 6)):
    ax.plot(eta, shapes[k, :, 0], lw=0.8, label=f'ring {k+1}')
ax.set_xlabel('η (deg)'); ax.set_ylabel('σ_G (px)')
ax.set_title('Pseudo-Voigt Gaussian width per (ring, η) — joint-cake refined')
ax.legend(ncols=3, fontsize=8); plt.tight_layout(); plt.show()

## 5. Compare to the alternating engine

Run `calibrate()` (the alternating engine + residual map) on the same image, table the refined parameters side-by-side. They should agree on geometry to within ~µε; differences indicate centroid bias that the joint engine corrected.

In [ ]:
from midas_calibrate_v2 import calibrate
alt = calibrate(img, wavelength=0.184139, pxY=150.0, dark=dark,
                calibrant='CeO2', verbose=False, output_dir=None)

print(f'{"param":<8s} {"joint cake":>14s} {"alternating":>14s} {"Δ":>10s}')
print('-' * 50)
for label, a, b in [
    ('Lsd', float(u['Lsd']), alt.Lsd),
    ('BC_y', float(u['BC_y']), alt.BC_y),
    ('BC_z', float(u['BC_z']), alt.BC_z),
    ('ty',  float(u['ty']), alt.ty),
    ('tz',  float(u['tz']), alt.tz),
]:
    print(f'{label:<8s} {a:14.4f} {b:14.4f} {a-b:+10.4f}')
print(f'\nalternating post-residual strain: {alt.post_residual_strain_uE:.1f} µε')
print(f'joint-cake cost:                   {result.cost:.4e}  '
      f'(different metric — sum of weighted intensity residuals)')

## 6. The cake itself

`result.cake` is a `CakeProfile` with the binned `(R, η, I)` array, the η bin centres, and the R bin centres. Useful for plotting + sanity-checking that the windows actually contain the rings.

In [ ]:
cake = result.cake
print(f'cake.intensity shape: {cake.intensity.shape}  '
      f'(n_R={cake.intensity.shape[0]}, n_eta={cake.intensity.shape[1]})')
fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(np.log1p(cake.intensity.clip(min=1)), origin='lower',
                aspect='auto', cmap='gray')
ax.set_xlabel('η bin'); ax.set_ylabel('R bin (px from BC)')
ax.set_title('Cake at joint-cake-refined geometry (log intensity)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## When joint-cake actually pays off

| symptom (after alternating + residual map) | likely cause | does joint-cake help? |
|---|---|---|
| Strain >100 µε, residual map already applied | Centroid bias from asymmetric peaks | **Yes** |
| Strain varies systematically with ring index | Per-ring shape variation not captured by uniform peak fitter | **Yes** |
| Doublet calibrant (Cu Kα), within-ring bias visible | Two unresolved peaks per ring | **Yes** — combine with NB 08's `fit_doublet_pairs` |
| Strain >100 µε but rings symmetric | Underconstrained geometry (NB 04, 07) | No — fix the gauge |
| Strain >100 µε and one ring is way out | Outlier ring contamination | No — fix the ring filter first |

Cost: joint-cake is ~5–10× slower than alternating because LBFGS over `(geometry + 6 × n_rings × n_eta)` DOFs is bigger than LM over just geometry + 15 distortion. Worth it only when alternating + residual map leaves >100 µε.